# Module 3: Workflow Agents (The Manager) 👔

Welcome to Module 3! We have built two specialist agents:
1.  **Financial Quant** (Module 1): Good at math and live data.
2.  **RAG Analyst** (Module 2): Good at reading internal documents.

Now we will build a **Manager Agent** that orchestrates them. This is a common **Multi-Agent Pattern** called "Routing".

### Learning Objectives
-   Import existing agents as Python modules.
-   Wrap agents as "Tools" for other agents.
-   Build a Manager that delegates tasks based on user intent.

## 1. Setup and Imports

We need to make sure we can import code from `module_01` and `module_02`. Since they are in sibling folders, we modify the Python path.

In [ ]:
%pip install "google-adk[mcp]" google-genai python-dotenv

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Load API Key
load_dotenv('../.env.local')

if "GOOGLE_API_KEY" not in os.environ:
    print("⚠️ Warning: GOOGLE_API_KEY not found.")
else:
    print("✅ API Key loaded.")

# Add project root to system path so we can import 'module_01' and 'module_02'
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project Root added to path: {project_root}")

## 2. Importing the Team

We import the `root_agent` from our previous modules. This is why we cleaned up the code into `agent.py` files!

In [ ]:
try:
    from module_01.financial_agent_app.agent import root_agent as financial_agent
    from module_02.rag_agent.agent import root_agent as rag_agent
    print("✅ Team recruited: Financial Agent and RAG Agent are ready.")
except ImportError as e:
    print(f"❌ Import Error: {e}")
    print("Make sure you have completed Module 1 and 2 and they have 'agent.py' files.")

## 3. Defining Delegation Tools

Agents don't magically talk to each other. We need to give the Manager a **Tool** that lets it "call" another agent.

We define functions that:
1.  Take a question from the Manager.
2.  Spin up a `Runner` for the sub-agent.
3.  Run the sub-agent and return the answer.

In [ ]:
from typing import Annotated
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types

async def ask_financial_quant(
    question: Annotated[str, "The financial question to ask (e.g. stock price, ROI)"]
) -> str:
    """Delegates a question to the Financial Quant agent."""
    print(f"   [Manager] Delegating to Financial Agent: '{question}'")
    
    runner = Runner(
        agent=financial_agent,
        app_name="financial_sub_task",
        session_service=InMemorySessionService(),
        auto_create_session=True
    )
    
    response_text = ""
    # We iterate through the response to get the final answer
    async for event in runner.run_async(
        user_id="manager",
        session_id="session_sub_1",
        new_message=types.Content(role="user", parts=[types.Part(text=question)])
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    
    return response_text

async def ask_rag_analyst(
    question: Annotated[str, "The document-based question to ask (policies, strategy)"]
) -> str:
    """Delegates a question to the RAG Analyst agent."""
    print(f"   [Manager] Delegating to RAG Agent: '{question}'")
    
    runner = Runner(
        agent=rag_agent,
        app_name="rag_sub_task",
        session_service=InMemorySessionService(),
        auto_create_session=True
    )
    
    response_text = ""
    async for event in runner.run_async(
        user_id="manager",
        session_id="session_sub_2",
        new_message=types.Content(role="user", parts=[types.Part(text=question)])
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    
    return response_text

## 4. The Manager Agent

Now we define the Manager. Its job is **Routing**.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import FunctionTool

manager_agent = Agent(
    model="gemini-3.5-flash-lite",
    name="manager_agent",
    tools=[
        FunctionTool(ask_financial_quant), 
        FunctionTool(ask_rag_analyst)
    ],
    instruction=(
        "You are a Senior Investment Manager. \n"
        "You have a team of experts:\n"
        "1. Financial Quant: Ask for real-time data like stock prices.\n"
        "2. RAG Analyst: Ask for internal knowledge like policies and strategy.\n\n"
        "Routing Rules:\n"
        "- If the user asks about 'price', 'value', or 'market data', ask the Financial Quant.\n"
        "- If the user asks about 'policy', 'strategy', 'guidelines', or 'outlook', ask the RAG Analyst.\n"
        "- Always synthesize the answer and present it professionally."
    )
)

print("✅ Manager Agent is ready to rule!")

## 5. Running the Manager (Interactive)

Try these questions:
1.  "What is the price of LUMR?" (Should go to Financial Quant)
2.  "What is our investment policy for tech?" (Should go to RAG Analyst)

In [ ]:
import uuid

runner = Runner(
    agent=manager_agent,
    app_name="manager_app",
    session_service=InMemorySessionService(),
    auto_create_session=True
)

async def run_chat():
    user_id = "user_manager"
    session_id = str(uuid.uuid4())
    print(f"starting session: {session_id}")

    while True:
        text = input("User (type 'quit' to exit): ")
        if text.lower() in ["quit", "exit"]:
            break
        
        print("   (Manager Thinking...)")
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=types.Content(role="user", parts=[types.Part(text=text)])
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"Manager: {part.text}")

# Uncomment to run in notebook
# await run_chat()

## 6. Going Production

Let's save this to `module_03/manager_agent/agent.py` so we can test and deploy it.

**Note:** The imports in the file need to be robust so they work when imported from outside.

In [ ]:
%%writefile manager_agent/agent.py
import os
import sys
import asyncio
from dotenv import load_dotenv
from typing import Annotated

from google.adk.agents import Agent
from google.adk.tools import FunctionTool
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types

# 1. Setup Environment & Imports
current_dir = os.path.dirname(os.path.abspath(__file__))
project_root = os.path.abspath(os.path.join(current_dir, "../../"))
env_path = os.path.join(project_root, ".env.local")
load_dotenv(env_path)

if "GOOGLE_API_KEY" not in os.environ:
    print(f"⚠️ Warning: GOOGLE_API_KEY not found in {env_path}")

# Add project root to path to allow importing sibling modules
if project_root not in sys.path:
    sys.path.append(project_root)

# Import the sub-agents
try:
    from module_01.financial_agent_app.agent import root_agent as financial_agent
    from module_02.rag_agent.agent import root_agent as rag_agent
    print("✅ Sub-agents imported successfully.")
except ImportError as e:
    print(f"❌ Failed to import sub-agents: {e}")

# 2. Define Delegation Tools

async def ask_financial_quant(
    question: Annotated[str, "The financial question to ask (e.g. stock price, ROI)"]
) -> str:
    """Delegates a question to the Financial Quant agent (Module 1)."""
    print(f"   [Manager] Delegating to Financial Agent: '{question}'")
    
    runner = Runner(
        agent=financial_agent,
        app_name="financial_sub_task",
        session_service=InMemorySessionService(),
        auto_create_session=True
    )
    
    response_text = ""
    async for event in runner.run_async(
        user_id="manager",
        session_id="session_1", 
        new_message=types.Content(role="user", parts=[types.Part(text=question)])
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    
    return response_text

async def ask_rag_analyst(
    question: Annotated[str, "The document-based question to ask (policies, strategy)"]
) -> str:
    """Delegates a question to the RAG Analyst agent (Module 2)."""
    print(f"   [Manager] Delegating to RAG Agent: '{question}'")
    
    runner = Runner(
        agent=rag_agent,
        app_name="rag_sub_task",
        session_service=InMemorySessionService(),
        auto_create_session=True
    )
    
    response_text = ""
    async for event in runner.run_async(
        user_id="manager",
        session_id="session_1",
        new_message=types.Content(role="user", parts=[types.Part(text=question)])
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    
    return response_text

# 3. Create the Manager Agent
root_agent = Agent(
    model="gemini-3.5-flash-lite",
    name="manager_agent",
    tools=[
        FunctionTool(ask_financial_quant), 
        FunctionTool(ask_rag_analyst)
    ],
    instruction=(
        "You are a Senior Investment Manager. \n"
        "You have a team of experts:\n"
        "1. Financial Quant: Ask for real-time data like stock prices.\n"
        "2. RAG Analyst: Ask for internal knowledge like policies and strategy.\n\n"
        "Routing Rules:\n"
        "- If the user asks about 'price', 'value', or 'market data', ask the Financial Quant.\n"
        "- If the user asks about 'policy', 'strategy', 'guidelines', or 'outlook', ask the RAG Analyst.\n"
        "- Always synthesize the answer and present it professionally."
    )
)